# OneStream LLM Spot Check

Stand-alone semi-manual notebook for the before-and-after Maersk LLM interface experiment.

## Run protocol

Set `RUN_LABEL` to `before` or `after`, load or generate `planning/spot_check_queries.csv`, and fill one result cell per query after copying the ticket into the Maersk LLM interface. The timestamped output CSV contains ticket IDs and grades, not raw ticket text.

In [1]:
# Imports
import os
import re
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Seed set to {SEED}")


Seed set to 42


In [2]:
# Configuration
RUN_LABEL = os.environ.get("ONESTREAM_SPOT_RUN_LABEL", "before")
if RUN_LABEL not in {"before", "after"}:
    raise ValueError("RUN_LABEL must be 'before' or 'after'")

SPOT_CHECK_QUERIES = Path("planning") / "spot_check_queries.csv"
OUTPUT_DIR = Path("dify_exports")
FIGURE_DIR = Path("assets") / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "RUN_LABEL": RUN_LABEL,
    "SPOT_CHECK_QUERIES": str(SPOT_CHECK_QUERIES),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "SEED": SEED,
}, indent=2))


{
  "RUN_LABEL": "after",
  "SPOT_CHECK_QUERIES": "planning/spot_check_queries.csv",
  "OUTPUT_DIR": "dify_exports",
  "SEED": 42
}


## Query set

The query file is fixed before grading. If it does not exist, this cell creates a 20-row template for Linus to fill in.

In [3]:
# Load or create query template
def redact_text(text, max_words=18):
    text = re.sub(r"\b[A-Z]{2,}[-_ ]?\d+\b", "internal_id", str(text))
    text = re.sub(r"\S+@\S+", "email", text)
    words = re.findall(r"[A-Za-z_]{2,}", text.lower())[:max_words]
    return " ".join(words)

def record_query_result(index, retrieved_document="", retrieved_chunk_excerpt="", final_llm_answer="", latency_seconds=np.nan, correctness="", usefulness="", notes=""):
    if index >= len(queries_df):
        print(f"Query {index + 1:02d} not present in query set; skipped")
        return None
    allowed_correctness = {"correct", "partial", "wrong", "none", "pending", ""}
    allowed_usefulness = {"resolved", "partial", "escalation", "pending", ""}
    correctness_value = correctness.strip().lower() if isinstance(correctness, str) else ""
    usefulness_value = usefulness.strip().lower() if isinstance(usefulness, str) else ""
    if correctness_value not in allowed_correctness:
        raise ValueError("correctness must be correct, partial, wrong, none, or blank")
    if usefulness_value not in allowed_usefulness:
        raise ValueError("usefulness must be resolved, partial, escalation, or blank")
    row = queries_df.iloc[index]
    record = {
        "run_label": RUN_LABEL,
        "ticket_id": row["ticket_id"],
        "ticket_text_redacted": redact_text(row.get("ticket_text", "")),
        "class_hint": row.get("class_hint", ""),
        "retrieved_document": retrieved_document.strip(),
        "retrieved_chunk_excerpt_redacted": redact_text(retrieved_chunk_excerpt, max_words=28),
        "final_llm_answer_redacted": redact_text(final_llm_answer, max_words=28) if final_llm_answer else "NO_ANSWER",
        "latency_seconds": latency_seconds,
        "correctness": correctness_value or "pending",
        "usefulness": usefulness_value or "pending",
        "notes": notes.strip(),
        "captured_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }
    manual_records.append(record)
    print(f"Recorded query {index + 1:02d}: {record['ticket_id']} as {record['correctness']} / {record['usefulness']}")
    return record

if not SPOT_CHECK_QUERIES.exists():
    SPOT_CHECK_QUERIES.parent.mkdir(parents=True, exist_ok=True)
    template = pd.DataFrame({
        "ticket_id": [f"Q{i:02d}" for i in range(1, 21)],
        "ticket_text": ["" for _ in range(20)],
        "class_hint": ["" for _ in range(20)],
        "notes": ["" for _ in range(20)],
    })
    template.to_csv(SPOT_CHECK_QUERIES, index=False)
    print(f"Created template at {SPOT_CHECK_QUERIES}. Linus should fill it in before the real run.")

queries_df = pd.read_csv(SPOT_CHECK_QUERIES).fillna("")
required_columns = ["ticket_id", "ticket_text", "class_hint", "notes"]
for column in required_columns:
    if column not in queries_df.columns:
        queries_df[column] = ""
queries_df = queries_df[required_columns].head(20).copy()
manual_records = []
print(f"Loaded {len(queries_df)} spot-check queries")
print(queries_df[["ticket_id", "class_hint", "notes"]].to_string(index=False))


Loaded 20 spot-check queries
ticket_id class_hint notes
      Q01                 
      Q02                 
      Q03                 
      Q04                 
      Q05                 
      Q06                 
      Q07                 
      Q08                 
      Q09                 
      Q10                 
      Q11                 
      Q12                 
      Q13                 
      Q14                 
      Q15                 
      Q16                 
      Q17                 
      Q18                 
      Q19                 
      Q20                 


## Manual capture cells

For each query, paste the ticket into the Maersk LLM UI, then fill the result fields in the matching cell. Blank fields are saved as `pending`, which lets the notebook run end-to-end before manual capture starts.

In [4]:
# Query 01 capture
record_query_result(
    0,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 01: Q01 as pending / pending


In [5]:
# Query 02 capture
record_query_result(
    1,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 02: Q02 as pending / pending


In [6]:
# Query 03 capture
record_query_result(
    2,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 03: Q03 as pending / pending


In [7]:
# Query 04 capture
record_query_result(
    3,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 04: Q04 as pending / pending


In [8]:
# Query 05 capture
record_query_result(
    4,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 05: Q05 as pending / pending


In [9]:
# Query 06 capture
record_query_result(
    5,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 06: Q06 as pending / pending


In [10]:
# Query 07 capture
record_query_result(
    6,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 07: Q07 as pending / pending


In [11]:
# Query 08 capture
record_query_result(
    7,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 08: Q08 as pending / pending


In [12]:
# Query 09 capture
record_query_result(
    8,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 09: Q09 as pending / pending


In [13]:
# Query 10 capture
record_query_result(
    9,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 10: Q10 as pending / pending


In [14]:
# Query 11 capture
record_query_result(
    10,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 11: Q11 as pending / pending


In [15]:
# Query 12 capture
record_query_result(
    11,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 12: Q12 as pending / pending


In [16]:
# Query 13 capture
record_query_result(
    12,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 13: Q13 as pending / pending


In [17]:
# Query 14 capture
record_query_result(
    13,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 14: Q14 as pending / pending


In [18]:
# Query 15 capture
record_query_result(
    14,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 15: Q15 as pending / pending


In [19]:
# Query 16 capture
record_query_result(
    15,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 16: Q16 as pending / pending


In [20]:
# Query 17 capture
record_query_result(
    16,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 17: Q17 as pending / pending


In [21]:
# Query 18 capture
record_query_result(
    17,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 18: Q18 as pending / pending


In [22]:
# Query 19 capture
record_query_result(
    18,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 19: Q19 as pending / pending


In [23]:
# Query 20 capture
record_query_result(
    19,
    retrieved_document="",
    retrieved_chunk_excerpt="",
    final_llm_answer="",
    latency_seconds=np.nan,
    correctness="",
    usefulness="",
    notes="",
)


Recorded query 20: Q20 as pending / pending


## Save run output

The output is timestamped and contains redacted excerpts only.

In [24]:
# Save timestamped run output
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_output_path = OUTPUT_DIR / f"spot_check_{RUN_LABEL}_{timestamp}.csv"
run_df = pd.DataFrame(manual_records)
if len(run_df) == 0:
    run_df = pd.DataFrame(columns=[
        "run_label", "ticket_id", "ticket_text_redacted", "class_hint", "retrieved_document",
        "retrieved_chunk_excerpt_redacted", "final_llm_answer_redacted", "latency_seconds",
        "correctness", "usefulness", "notes", "captured_at"
    ])
run_df.to_csv(run_output_path, index=False)
print(f"Wrote {run_output_path}")
print(run_df[["ticket_id", "correctness", "usefulness"]].head(20).to_string(index=False))


Wrote dify_exports/spot_check_after_20260507_022001.csv
ticket_id correctness usefulness
      Q01     pending    pending
      Q02     pending    pending
      Q03     pending    pending
      Q04     pending    pending
      Q05     pending    pending
      Q06     pending    pending
      Q07     pending    pending
      Q08     pending    pending
      Q09     pending    pending
      Q10     pending    pending
      Q11     pending    pending
      Q12     pending    pending
      Q13     pending    pending
      Q14     pending    pending
      Q15     pending    pending
      Q16     pending    pending
      Q17     pending    pending
      Q18     pending    pending
      Q19     pending    pending
      Q20     pending    pending


## Before-and-after analysis

After both timestamped runs exist, this cell computes aggregate transitions, a side-by-side ticket table, and two trace figures.

In [25]:
# Analyze before-and-after runs
def latest_run(label):
    files = sorted(OUTPUT_DIR.glob(f"spot_check_{label}_*.csv"))
    return files[-1] if files else None

def correctness_rank(value):
    order = {"none": 0, "pending": 0, "wrong": 1, "partial": 2, "correct": 3}
    return order.get(str(value).lower(), 0)

def usefulness_rank(value):
    order = {"escalation": 0, "pending": 0, "partial": 1, "resolved": 2}
    return order.get(str(value).lower(), 0)

before_path = latest_run("before")
after_path = latest_run("after")
if before_path is None or after_path is None:
    print("[skip] reason: both before and after run CSVs are required for aggregate comparison")
    comparison_df = pd.DataFrame()
    transitions_df = pd.DataFrame()
else:
    before_df = pd.read_csv(before_path).fillna("")
    after_df = pd.read_csv(after_path).fillna("")
    comparison_df = before_df.merge(after_df, on="ticket_id", suffixes=("_before", "_after"), how="outer")
    comparison_df["correctness_transition"] = comparison_df["correctness_before"].astype(str) + " -> " + comparison_df["correctness_after"].astype(str)
    comparison_df["usefulness_transition"] = comparison_df["usefulness_before"].astype(str) + " -> " + comparison_df["usefulness_after"].astype(str)
    transitions = {
        "no_answer_to_answered": int(((comparison_df["correctness_before"].isin(["none", "pending"])) & (comparison_df["correctness_after"].isin(["partial", "correct"]))).sum()),
        "wrong_to_correct": int(((comparison_df["correctness_before"] == "wrong") & (comparison_df["correctness_after"] == "correct")).sum()),
        "partial_to_correct": int(((comparison_df["correctness_before"] == "partial") & (comparison_df["correctness_after"] == "correct")).sum()),
        "regressions": int((comparison_df.apply(lambda r: correctness_rank(r.get("correctness_after", "")) < correctness_rank(r.get("correctness_before", "")), axis=1)).sum()),
    }
    transitions_df = pd.DataFrame([transitions])
    comparison_path = OUTPUT_DIR / "spot_check_comparison_latest.csv"
    transitions_path = OUTPUT_DIR / "spot_check_transitions_latest.csv"
    comparison_df.to_csv(comparison_path, index=False)
    transitions_df.to_csv(transitions_path, index=False)
    print(transitions_df.to_string(index=False))
    print(comparison_df[["ticket_id", "correctness_before", "correctness_after", "usefulness_before", "usefulness_after"]].to_string(index=False))

    plot_rows = comparison_df.head(2).copy()
    plt.rcParams.update({"font.family": "serif", "figure.facecolor": "white", "axes.facecolor": "white"})
    for idx, (_, row) in enumerate(plot_rows.iterrows(), start=1):
        fig, ax = plt.subplots(figsize=(6.4, 1.8))
        before_score = correctness_rank(row.get("correctness_before", ""))
        after_score = correctness_rank(row.get("correctness_after", ""))
        ax.barh(["before", "after"], [before_score, after_score], color=["#6c757d", "#4967AA"])
        ax.set_xlim(0, 3)
        ax.set_xlabel("Correctness grade")
        ax.set_yticks([0, 1], labels=["before", "after"])
        ax.grid(axis="x", color="#e9ecef")
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"fig_spot_check_trace_{idx}.png", dpi=300, bbox_inches="tight", facecolor="white")
        fig.savefig(FIGURE_DIR / f"fig_spot_check_trace_{idx}.pdf", dpi=300, bbox_inches="tight", facecolor="white")
        plt.close(fig)


 no_answer_to_answered  wrong_to_correct  partial_to_correct  regressions
                     0                 0                   0            0
ticket_id correctness_before correctness_after usefulness_before usefulness_after
      Q01            pending           pending           pending          pending
      Q02            pending           pending           pending          pending
      Q03            pending           pending           pending          pending
      Q04            pending           pending           pending          pending
      Q05            pending           pending           pending          pending
      Q06            pending           pending           pending          pending
      Q07            pending           pending           pending          pending
      Q08            pending           pending           pending          pending
      Q09            pending           pending           pending          pending
      Q10            pending    